# Sentiment Analysis using NLP Pipeline & ML Models

## Objective
Build an end-to-end sentiment analysis system using NLP preprocessing, feature engineering, and machine learning models.

## Dataset
Dataset used: IMDb Reviews Dataset (Kaggle)

## Pipeline
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison


In [15]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
df = pd.read_csv("dataset.csv") 

df.head()


,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [17]:
print(df.columns)


Index(['text', 'label'], dtype='str')


In [18]:
print("Shape:", df.shape)
print("\nClass Distribution:\n", df['label'].value_counts())

df['text'][0]


Shape: (40000, 2)

Class Distribution:
 label
0    20019
1    19981
Name: count, dtype: int64


'I grew up (b. 1965) watching and loving the Thunderbirds. All my mates at school watched. We played "Thunderbirds" before school, during lunch and after school. We all wanted to be Virgil or Scott. No one wanted to be Alan. Counting down from 5 became an art form. I took my children to see the movie hoping they would get a glimpse of what I loved as a child. How bitterly disappointing. The only high point was the snappy theme tune. Not that it could compare with the original score of the Thunderbirds. Thankfully early Saturday mornings one television channel still plays reruns of the series Gerry Anderson and his wife created. Jonatha Frakes should hand in his directors chair, his version was completely hopeless. A waste of film. Utter rubbish. A CGI remake may be acceptable but replacing marionettes with Homo sapiens subsp. sapiens was a huge error of judgment.'

In [19]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'\d+', '', text)      # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()


,text,clean_text
0,I grew up (b. 1965) watching and loving the Th...,grew b watching loving thunderbird mate school...
1,"When I put this movie in my DVD player, and sa...",put movie dvd player sat coke chip expectation...
2,Why do people who do not know what a particula...,people know particular time past like feel nee...
3,Even though I have great interest in Biblical ...,even though great interest biblical movie bore...
4,Im a die hard Dads Army fan and nothing will e...,im die hard dad army fan nothing ever change g...


In [20]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])
y = df['label']


In [21]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)


In [22]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

y_pred_dt = dt.predict(X_test_bow)


In [23]:
def evaluate(y_test, y_pred, model_name):
    print(f"\n--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")



--- Logistic Regression ---
Accuracy: 0.8845
Precision: 0.8845522418988176
Recall: 0.8845
F1 Score: 0.8844887362156485

--- Naive Bayes ---
Accuracy: 0.842625
Precision: 0.842678975015436
Recall: 0.842625
F1 Score: 0.842627643434018

--- Decision Tree ---
Accuracy: 0.711625
Precision: 0.7117402294774396
Recall: 0.711625
F1 Score: 0.7116233553606857


In [25]:
print("Logistic Regression")
evaluate(y_test, y_pred_lr, "Logistic Regression")

print("\nNaive Bayes")
evaluate(y_test, y_pred_nb, "Naive Bayes")

print("\nDecision Tree")
evaluate(y_test, y_pred_dt, "Decision Tree")


Logistic Regression

--- Logistic Regression ---
Accuracy: 0.8845
Precision: 0.8845522418988176
Recall: 0.8845
F1 Score: 0.8844887362156485

Naive Bayes

--- Naive Bayes ---
Accuracy: 0.842625
Precision: 0.842678975015436
Recall: 0.842625
F1 Score: 0.842627643434018

Decision Tree

--- Decision Tree ---
Accuracy: 0.711625
Precision: 0.7117402294774396
Recall: 0.711625
F1 Score: 0.7116233553606857


## Model Comparison & Insights

- Logistic Regression performed best with TF-IDF features.
- Naive Bayes worked well with Bag of Words due to probability assumptions.
- Decision Tree showed lower performance due to overfitting.

## Best Preprocessing Steps
- Removing stopwords improved clarity
- Lemmatization helped reduce word variations

## Best Feature Engineering
- TF-IDF performed better than BoW because it considers importance of words

## Trade-offs
- Naive Bayes → Fast but less accurate
- Logistic Regression → Balanced performance
- Decision Tree → Easy to interpret but overfits

## Conclusion
TF-IDF + Logistic Regression is the best combination for sentiment analysis in this task.


## Final Conclusion

This project demonstrated how raw text is Best Model: Logistic Regression
Best Feature: TF-IDF
Why?
Handles word importance better
Trade-off:
Naive Bayes → fast
Decision Tree → overfitting
